In [171]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [172]:
df_train_cleaned = pd.read_csv("../data/train_cleaned.csv")
df_train_cleaned.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


In [ ]:
# distance speed
df_train_cleaned["distance_speed"] = df_train_cleaned["dist_min_ci_0_5h"] / (df_train_cleaned["area_first_ha"] + 1)

# direction alignment
df_train_cleaned["direction_alignment"] = df_train_cleaned["spread_bearing_cos"] * df_train_cleaned["alignment_cos"]


# direction alignment
df_train_cleaned["direction_alignment"] = df_train_cleaned["spread_bearing_cos"] * df_train_cleaned["alignment_cos"]


df_train_cleaned["fire_pressure"] = df_train_cleaned["area_first_ha"] / (df_train_cleaned["dist_min_ci_0_5h"] + 1)


In [174]:
df_train_cleaned.columns

Index(['event_id', 'num_perimeters_0_5h', 'dt_first_last_0_5h',
       'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h',
       'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first',
       'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h',
       'radial_growth_m', 'radial_growth_rate_m_per_h',
       'centroid_displacement_m', 'centroid_speed_m_per_h',
       'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos',
       'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h',
       'dist_slope_ci_0_5h', 'closing_speed_m_per_h',
       'closing_speed_abs_m_per_h', 'projected_advance_m',
       'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos',
       'alignment_abs', 'cross_track_component', 'along_track_speed',
       'event_start_hour', 'event_start_dayofweek', 'event_start_month',
       'time_to_hit_hours', 'event', 'distance_speed', 'direction_alignment',
       'fire_pressure'],
      dtype='object')

In [175]:
# ------------------------------------------------
# IDENTIFIER
# ------------------------------------------------

ID_COL = ["event_id"]

# ------------------------------------------------
# TARGET
# ------------------------------------------------

TARGET_TIME = ["time_to_hit_hours"]
TARGET_EVENT = ["event"]

# ------------------------------------------------
# CATEGORICAL / BINARY
# ------------------------------------------------

categorical = [
    "low_temporal_resolution_0_5h"
]

# ------------------------------------------------
# CATEGORICAL COUNT
# ------------------------------------------------

categorical_count = [
    "num_perimeters_0_5h"
]

# ------------------------------------------------
# TIME FEATURES
# ------------------------------------------------

time_features = [
    "dt_first_last_0_5h",
    "event_start_hour",
    "event_start_dayofweek",
    "event_start_month"
]

# ------------------------------------------------
# CONTINUOUS FEATURES
# ------------------------------------------------

continuous = [

    # fire size / growth
    "area_first_ha",
    "area_growth_abs_0_5h",
    "area_growth_rel_0_5h",
    "area_growth_rate_ha_per_h",
    "log1p_area_first",
    "log1p_growth",
    "log_area_ratio_0_5h",
    "relative_growth_0_5h",

    # radial growth
    "radial_growth_m",
    "radial_growth_rate_m_per_h",

    # centroid movement
    "centroid_displacement_m",
    "centroid_speed_m_per_h",

    # spread direction
    "spread_bearing_deg",
    "spread_bearing_sin",
    "spread_bearing_cos",

    # distance to evacuation zone
    "dist_min_ci_0_5h",
    "dist_std_ci_0_5h",
    "dist_change_ci_0_5h",
    "dist_slope_ci_0_5h",

    # closing speed
    "closing_speed_m_per_h",
    "closing_speed_abs_m_per_h",
    "projected_advance_m",

    # distance dynamics
    "dist_accel_m_per_h2",
    "dist_fit_r2_0_5h",

    # direction alignment
    "alignment_cos",
    "alignment_abs",

    # vector movement
    "cross_track_component",
    "along_track_speed",

    # engineered features
    "distance_speed",
    "direction_alignment",
    "fire_pressure"
]

# ------------------------------------------------
# ALL FEATURES
# ------------------------------------------------

ALL_FEATURES = (
    continuous
    + categorical
    + categorical_count
    + time_features
)

print("Continuous:", len(continuous))
print("Categorical:", len(categorical))
print("Categorical Count:", len(categorical_count))
print("Time Features:", len(time_features))
print("Total Features:", len(ALL_FEATURES))

Continuous: 31
Categorical: 1
Categorical Count: 1
Time Features: 4
Total Features: 37


In [176]:
df_train_cleaned[continuous].std().sort_values()

fire_pressure                      0.121856
dist_fit_r2_0_5h                   0.171690
spread_bearing_sin                 0.285193
log_area_ratio_0_5h                0.300321
direction_alignment                0.310445
alignment_abs                      0.329210
spread_bearing_cos                 0.351904
alignment_cos                      0.371909
area_growth_rel_0_5h               1.302001
relative_growth_0_5h               1.302001
log1p_growth                       1.340348
log1p_area_first                   2.083529
dist_accel_m_per_h2               11.088141
closing_speed_abs_m_per_h         26.690409
closing_speed_m_per_h             26.865184
cross_track_component             37.789199
radial_growth_rate_m_per_h        37.840514
area_growth_rate_ha_per_h         40.467370
dist_slope_ci_0_5h                41.511198
spread_bearing_deg                46.703309
along_track_speed                 46.760648
centroid_speed_m_per_h            58.940466
dist_std_ci_0_5h                

### Variance Analysis

The standard deviation of all continuous features was examined to identify potential low-variance variables.

None of the features exhibited extremely low variance (close to zero).  
Therefore, no predictors were removed based on the low-variance criterion.

All remaining features retain sufficient variability and may contribute meaningful information to the predictive model.

In [177]:
corr = df_train_cleaned[continuous].corrwith(df_train_cleaned["event"])
corr.sort_values()

dist_min_ci_0_5h             -0.481379
spread_bearing_cos           -0.323189
area_first_ha                -0.181334
log1p_area_first             -0.167912
distance_speed               -0.139257
dist_slope_ci_0_5h           -0.115274
dist_change_ci_0_5h          -0.106449
dist_accel_m_per_h2          -0.072594
cross_track_component        -0.058287
along_track_speed             0.008147
direction_alignment           0.093941
projected_advance_m           0.106449
closing_speed_m_per_h         0.107393
alignment_cos                 0.119933
closing_speed_abs_m_per_h     0.138735
dist_std_ci_0_5h              0.141964
dist_fit_r2_0_5h              0.143101
area_growth_abs_0_5h          0.158327
relative_growth_0_5h          0.165975
area_growth_rel_0_5h          0.165975
area_growth_rate_ha_per_h     0.172416
spread_bearing_sin            0.188252
centroid_displacement_m       0.207992
centroid_speed_m_per_h        0.209254
radial_growth_m               0.209343
radial_growth_rate_m_per_

In [178]:
# drop_features = [
#     "along_track_speed",
#     "cross_track_component",
#     "dist_accel_m_per_h2"
# ]
# df_train_cleaned = df_train_cleaned.drop(columns=drop_features)

In [179]:
corr_time = df_train_cleaned[continuous].corrwith(df_train_cleaned["time_to_hit_hours"])

corr_time.sort_values()

alignment_abs                -0.366923
spread_bearing_deg           -0.335598
log1p_growth                 -0.317451
fire_pressure                -0.311948
dist_fit_r2_0_5h             -0.284143
log_area_ratio_0_5h          -0.228119
radial_growth_rate_m_per_h   -0.218137
radial_growth_m              -0.217379
centroid_displacement_m      -0.212354
spread_bearing_sin           -0.211706
centroid_speed_m_per_h       -0.208943
area_growth_rate_ha_per_h    -0.177191
closing_speed_abs_m_per_h    -0.166876
area_growth_abs_0_5h         -0.164590
dist_std_ci_0_5h             -0.160923
area_growth_rel_0_5h         -0.157423
relative_growth_0_5h         -0.157423
projected_advance_m          -0.131773
closing_speed_m_per_h        -0.130362
alignment_cos                -0.079847
direction_alignment          -0.066657
along_track_speed            -0.023527
distance_speed               -0.001626
cross_track_component         0.052684
dist_slope_ci_0_5h            0.125267
dist_change_ci_0_5h      

In [180]:
corr_event = df_train_cleaned[continuous].corrwith(df_train_cleaned["event"])
corr_time = df_train_cleaned[continuous].corrwith(df_train_cleaned["time_to_hit_hours"])

low_signal = []

for col in continuous:
    
    if abs(corr_event[col]) < 0.05 and abs(corr_time[col]) < 0.05:
        low_signal.append(col)

print("Low signal features:", low_signal)

Low signal features: ['along_track_speed']


In [181]:
# Drop Those Features

df_train_cleaned = df_train_cleaned.drop(columns=["along_track_speed"])

In [182]:
# Check Remaining Features
continuous.remove("along_track_speed")
print("Remaining features:", df_train_cleaned.columns)

Remaining features: Index(['event_id', 'num_perimeters_0_5h', 'dt_first_last_0_5h',
       'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h',
       'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first',
       'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h',
       'radial_growth_m', 'radial_growth_rate_m_per_h',
       'centroid_displacement_m', 'centroid_speed_m_per_h',
       'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos',
       'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h',
       'dist_slope_ci_0_5h', 'closing_speed_m_per_h',
       'closing_speed_abs_m_per_h', 'projected_advance_m',
       'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos',
       'alignment_abs', 'cross_track_component', 'event_start_hour',
       'event_start_dayofweek', 'event_start_month', 'time_to_hit_hours',
       'event', 'distance_speed', 'direction_alignment', 'fire_pressure'],
      dtype='object')


In [183]:
X = df_train_cleaned.drop(columns=["event_id", "time_to_hit_hours", "event"])

# VIF Varience Inflation Factor

How much 1 independent variable impacts otheer independent variables<br>
if VIF < 10 (thresold values) then teir exist a problem
if VIF < 10 for any independent values then drop that particular variables

In [184]:
features = [col for col in continuous if col in df_train_cleaned.columns]

X = df_train_cleaned[features]

from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

vif.sort_values("VIF", ascending=False)

c:\Users\ibtes\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
2,area_growth_rel_0_5h,inf
7,relative_growth_0_5h,inf
21,projected_advance_m,inf
17,dist_change_ci_0_5h,inf
3,area_growth_rate_ha_per_h,7.059565e+04
1,area_growth_abs_0_5h,6.622483e+04
9,radial_growth_rate_m_per_h,2.444424e+04
19,closing_speed_m_per_h,2.428067e+04
16,dist_std_ci_0_5h,2.248799e+04
18,dist_slope_ci_0_5h,2.008377e+04


In [185]:
drop_vif = [
    "area_growth_rel_0_5h",
    "relative_growth_0_5h",
    "projected_advance_m",
    "dist_change_ci_0_5h"
]

df_train_cleaned = df_train_cleaned.drop(columns=drop_vif)

In [186]:
features = [col for col in continuous if col in df_train_cleaned.columns]

X = df_train_cleaned[features]
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

vif.sort_values("VIF", ascending=False)

,feature,VIF
2,area_growth_rate_ha_per_h,24796.346094
1,area_growth_abs_0_5h,22531.801564
16,closing_speed_m_per_h,17571.650600
15,dist_slope_ci_0_5h,14145.867038
7,radial_growth_rate_m_per_h,13218.537060
6,radial_growth_m,12162.941282
8,centroid_displacement_m,10290.681520
9,centroid_speed_m_per_h,5697.114902
14,dist_std_ci_0_5h,3275.951233
17,closing_speed_abs_m_per_h,1433.015183


In [187]:
drop_growth = [
    "area_growth_rate_ha_per_h",
    "area_growth_abs_0_5h",
    "radial_growth_rate_m_per_h",
    "radial_growth_m"
]
drop_motion = [
    "centroid_displacement_m"
]

drop_distance = [
    "dist_std_ci_0_5h",
    "dist_slope_ci_0_5h"
]

drop_direction = [
    "spread_bearing_cos",
    "spread_bearing_sin"
]

drop_alignment = [
    "alignment_cos",
    "direction_alignment"
]


drop_cols = (
    drop_growth
    + drop_motion
    + drop_distance
    + drop_direction
    + drop_alignment
)

df_train_cleaned = df_train_cleaned.drop(columns=drop_cols)

In [188]:
features = [col for col in continuous if col in df_train_cleaned.columns]

X = df_train_cleaned[features]
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

vif.sort_values("VIF", ascending=False)

,feature,VIF
7,closing_speed_m_per_h,21.982839
3,log_area_ratio_0_5h,16.868219
4,centroid_speed_m_per_h,16.056722
8,closing_speed_abs_m_per_h,15.780913
2,log1p_growth,9.891130
9,dist_accel_m_per_h2,7.350462
10,dist_fit_r2_0_5h,3.822746
5,spread_bearing_deg,3.783894
1,log1p_area_first,3.485885
6,dist_min_ci_0_5h,2.129042


In [189]:
drop_cols = [
"closing_speed_m_per_h",
"closing_speed_abs_m_per_h"
]

df_train_cleaned = df_train_cleaned.drop(columns=drop_cols)

In [190]:
features = [col for col in continuous if col in df_train_cleaned.columns]

X = df_train_cleaned[features]
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

vif.sort_values("VIF", ascending=False)

,feature,VIF
3,log_area_ratio_0_5h,11.768985
4,centroid_speed_m_per_h,9.553574
2,log1p_growth,8.493324
5,spread_bearing_deg,3.584371
1,log1p_area_first,3.474560
8,dist_fit_r2_0_5h,3.190601
7,dist_accel_m_per_h2,2.284998
6,dist_min_ci_0_5h,2.118400
12,fire_pressure,1.825897
10,cross_track_component,1.646165


### Final Multicollinearity Check

Variance Inflation Factor (VIF) was recalculated after removing highly collinear variables.  
Most features now have VIF values below 10, indicating acceptable multicollinearity levels.

A few growth-related features show slightly higher VIF values due to their inherent relationship with fire expansion dynamics. However, these variables were retained because they capture important wildfire growth behavior.

The final feature set provides a balance between reducing redundancy and preserving meaningful predictive information.

In [191]:
df_train_cleaned

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,log1p_area_first,log1p_growth,log_area_ratio_0_5h,centroid_speed_m_per_h,spread_bearing_deg,...,dist_fit_r2_0_5h,alignment_abs,cross_track_component,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event,distance_speed,fire_pressure
0,10892457,3,4.265188,0,79.696304,4.390693,1.354787,0.03545,1.940119,70.130507,...,0.886373,0.054649,-1.937219,19,4,5,18.892512,0,76.411450,0.012923
1,11757157,2,1.169918,0,8.946749,2.297246,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.568898,-0.000000,4,4,6,22.048108,1,294.661687,0.003051
2,11945086,4,4.777526,0,106.482638,4.677329,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.882385,0.000000,22,4,8,0.888895,1,30.445616,0.032530
3,12044083,1,0.000000,1,67.631125,4.228746,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,20,5,8,60.953021,0,934.268113,0.001055
4,12052347,2,4.975273,0,35.632874,3.600946,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.934634,-0.000000,21,5,7,44.990274,0,491.510235,0.001979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,97075632,1,0.000000,1,51.295195,3.956904,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,17,6,6,66.340624,0,5358.660441,0.000183
217,97362560,2,1.127102,0,1.176991,0.777943,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.277779,-0.000000,18,1,7,5.694898,1,2087.721005,0.000259
218,97805715,2,3.710653,0,71.946930,4.289732,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.694609,0.000000,18,1,9,44.011253,0,82.685801,0.011926
219,99071478,1,0.000000,1,20.223659,3.055117,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,15,0,8,22.975783,1,220.219066,0.004326


In [192]:
# Drop Event ID
df_train_cleaned = df_train_cleaned.drop(columns=["event_id"])

In [193]:
# Save Data
df_train_cleaned.to_csv("../data/train_feature_selected.csv", index=False)